# NumCompute ? Streaming Machine Learning Demo

This notebook demonstrates how to process data in chunks, train a streaming decision tree, track performance, and visualise results.

The example uses the supplied customer churn dataset.

## 1. Import libraries and project modules

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from numcompute.io import iter_csv_chunks
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.visualise import plot_metric_over_time

## 2. Configure the streaming model

We use age and income as numerical input features and churn as the target. The city and plan columns are excluded from this demonstration.

In [ ]:
data_path = Path('sample_data.csv')
if not data_path.exists():
    data_path = Path('demo/sample_data.csv')

model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    buffer_size=32,
    random_state=42
)

trainer = StreamTrainer(model)

print('Model configured successfully.')

## 3. Read the CSV file in chunks

The custom CSV reader loads a small portion of the dataset at a time. Missing numeric values are represented as NaN.

In [ ]:
chunks = iter_csv_chunks(
    data_path,
    chunksize=16,
    skip_header=1,
    usecols=[0, 1, 4],
    missing='nan'
)

print('CSV reader ready.')

## 4. Train and evaluate incrementally

Each chunk is evaluated before it is used for training. This is called prequential evaluation. The decision tree rebuilds when its internal buffer reaches the configured size.

In [ ]:
for chunk in chunks:
    X = chunk[:, :2]
    y = chunk[:, 2].astype(int)
    trainer.process_chunk(X, y)

print('Streaming training completed.')
print('Chunks processed:', trainer.summary()['n_chunks'])
print('Samples processed:', trainer.summary()['n_samples_seen'])
print('Tree retraining events:', trainer.summary()['n_retrains'])

## 5. Review streaming performance

In [ ]:
summary = trainer.summary()

for metric, value in summary.items():
    print(f'{metric}: {value}')

## 6. Visualise cumulative accuracy

In [ ]:
accuracy_values = [
    row['cumulative_accuracy']
    for row in trainer.history
    if row['cumulative_accuracy'] is not None
]

if accuracy_values:
    plot_metric_over_time(
        accuracy_values,
        title='Streaming Decision Tree ? Cumulative Accuracy',
        ylabel='Accuracy'
    )
else:
    print('No predictions were available to plot.')

## 7. Conclusion

This demonstration shows how NumCompute processes a CSV dataset incrementally rather than loading the entire dataset into the model at once. It also demonstrates buffered decision-tree retraining, prequential evaluation, performance tracking, and visualisation.